# Uplift (X5 Retail Hero): S-learner + честная оценка

**Задача.** Промо-SMS в среднем поднимает вероятность покупки на **+3.3 п.п.** (контроль 60.3% →
лечение 63.7%). Но рассылать всем невыгодно: кого-то SMS убеждает, кто-то купил бы и так («sure things»),
кого-то даже отталкивает. Uplift-модель ранжирует клиентов по **индивидуальному** приросту, чтобы
таргетировать только тех, кого SMS реально двигает.

**Что было не так.** В исходном проекте (1) базовые модели были почти случайны (AUC покупки ~0.53) из-за
недообученного CatBoost, поэтому uplift = разность двух near-random вероятностей → узкий шум («модель
ничего не предсказывает»); (2) качество мерили MAE против файла `uplift_sub.csv` — а это **прокси**, не
настоящий ground truth (индивидуального uplift на тесте физически не существует). Настоящая метрика
(Qini / uplift@k по реальному A/B) была импортирована, но не использовалась.

**Здесь — как надо:** сильный `HistGradientBoostingClassifier`, S-learner, и честный Qini/uplift@k по
реальным `treatment_flg`/`target` на отложенной выборке. Продакшн: `backend/training/uplift.py` +
serve-обёртка `app/uplift.py::SoloModelUplift`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("../../data/uplift/train.csv")
NUM = ["age","issue_redeem_days_diff","total_purchase_count","total_purchase_sum","avg_purchase_sum",
       "avg_days_between_purchases","purchase_frequency_monthly","unique_products",
       "alcohol_purchase_ratio","private_label_purchase_ratio","growth_rate_last_3_months","count_above_1000"]
CAT = ["gender"]
FEATURES = NUM + CAT
w = df["treatment_flg"].to_numpy(); y = df["target"].to_numpy()
print(f"строк: {len(df)} | лечение/контроль: {int(w.sum())}/{int((1-w).sum())}")
print(f"ATE = {y[w==1].mean() - y[w==0].mean():+.4f}  ({y[w==0].mean():.3f} -> {y[w==1].mean():.3f})")

строк: 200039 | лечение/контроль: 99981/100058
ATE = +0.0332  (0.603 -> 0.637)


## S-learner (Treatment Dummy)

Одна модель на `[фичи + treatment_flg]`. Для клиента считаем `P(покупка|SMS=1) − P(покупка|SMS=0)` — два
прогона того же классификатора. Простой, устойчивый, и естественно ложится в сервинг (один артефакт).

In [ ]:
def make():
    pre = ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM + ["treatment_flg"]),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    clf = HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05, max_leaf_nodes=31,
                                         min_samples_leaf=50, l2_regularization=1.0, random_state=42)
    return Pipeline([("pre", pre), ("clf", clf)])

Xtr, Xval, wtr, wval, ytr, yval = train_test_split(df[FEATURES], w, y, test_size=0.25,
                                                    random_state=42, stratify=w*2+y)
model = make().fit(Xtr.assign(treatment_flg=wtr), ytr)

def p_at(flag):
    return model.predict_proba(Xval.assign(treatment_flg=flag))[:, 1]
uplift = p_at(1) - p_at(0)
# base purchase AUC — прогон с фактическим флагом лечения (для sanity: сигнал покупки есть)
base_auc = roc_auc_score(yval, model.predict_proba(Xval.assign(treatment_flg=wval))[:, 1])
print("base purchase AUC (val):", round(base_auc, 4))
print(f"uplift score: std={uplift.std():.4f}  p10={np.percentile(uplift,10):+.4f}  "
      f"p50={np.percentile(uplift,50):+.4f}  p90={np.percentile(uplift,90):+.4f}  "
      f"min={uplift.min():+.4f}  max={uplift.max():+.4f}")

base purchase AUC (val): 0.7601
uplift score: std=0.0311  p10=+0.0000  p50=+0.0211  p90=+0.0660  min=-0.0729  max=+0.3179


## Честная оценка по A/B (не против прокси-файла)

`uplift@k` — прирост среди top-k% по предсказанному скору (отклик лечения минус контроля). Qini —
нормированная площадь Qini-кривы: 0 = случайный таргетинг, 1 = идеальное ранжирование.

In [ ]:
def uplift_at_k(y, w, s, k):
    idx = np.argsort(-s)[:int(np.ceil(k*len(s)))]
    return y[idx][w[idx]==1].mean() - y[idx][w[idx]==0].mean()

def qini_area(y, w, order):
    y_, w_ = y[order].astype(float), w[order].astype(float)
    nt, nc = np.cumsum(w_), np.cumsum(1-w_)
    with np.errstate(divide="ignore", invalid="ignore"):
        c = np.cumsum(y_*w_) - np.cumsum(y_*(1-w_))*np.where(nc>0, nt/nc, 0.0)
    c = np.concatenate([[0.0], c]); x = np.arange(len(c))
    return np.trapezoid(c - x/x[-1]*c[-1], x)

def qini(y, w, s):
    perfect = np.argsort(-np.where(w==1, y+1, -(1-y)))
    return qini_area(y, w, np.argsort(-s)) / qini_area(y, w, perfect)

print(f"ATE(val)     = {yval[wval==1].mean() - yval[wval==0].mean():+.4f}")
for k in (0.1, 0.2, 0.3):
    print(f"uplift@{int(k*100)}%   = {uplift_at_k(yval, wval, uplift, k):+.4f}")
print(f"Qini coef    = {qini(yval, wval, uplift):.4f}   (случайный скор ~{qini(yval, wval, np.random.default_rng(0).random(len(uplift))):+.4f})")

ATE(val)     = +0.0332
uplift@10%   = +0.0989
uplift@20%   = +0.0710
uplift@30%   = +0.0567
Qini coef    = 0.0818   (случайный скор ~+0.0087)


## Скор действительно варьируется (не «плоский»)

Разложим val по децилям предсказанного uplift и посмотрим фактический отклик лечение/контроль в каждом.
Верхний дециль должен концентрировать прирост, нижние — около нуля или отрицательны («спящие»/«не буди»).
Наблюдаемый uplift по одному децилю шумит (это разность двух ~50% откликов на подвыборке ~5k), поэтому
смотрим на края и агрегаты (Qini / uplift@k), а не на идеальную монотонность каждого дециля.

In [ ]:
q = pd.qcut(uplift, 10, labels=False, duplicates="drop")
rows = []
for d in sorted(pd.unique(q), reverse=True):
    m = q == d
    rt = yval[m & (wval==1)].mean(); rc = yval[m & (wval==0)].mean()
    rows.append({"decile(top→bottom)": int(9-d)+1, "pred_uplift": round(uplift[m].mean(),4),
                 "resp_treat": round(rt,3), "resp_control": round(rc,3), "obs_uplift": round(rt-rc,4),
                 "n": int(m.sum())})
print(pd.DataFrame(rows).to_string(index=False))

 decile(top→bottom)  pred_uplift  resp_treat  resp_control  obs_uplift    n
                  1       0.0981       0.533         0.434      0.0989 5001
                  2       0.0555       0.531         0.487      0.0442 5001
                  3       0.0413       0.582         0.556      0.0265 5001
                  4       0.0316       0.592         0.565      0.0264 5001
                  5       0.0243       0.633         0.615      0.0182 5001
                  6       0.0181       0.660         0.624      0.0366 5001
                  7       0.0125       0.699         0.657      0.0422 5001
                  8       0.0073       0.745         0.715      0.0307 5001
                  9       0.0025       0.725         0.736     -0.0101 5001
                 10      -0.0063       0.668         0.639      0.0297 5001


## Итог

| | Значение |
|---|---|
| base purchase AUC | **≈0.76** (у исходника было ~0.53) |
| **Qini coefficient** | **≈0.08** (случайный ~0) |
| uplift@10% / @20% / @30% | **≈9.9 / 7.1 / 5.7 п.п.** при среднем ATE 3.3 п.п. |
| разброс скора | не плоский (см. std и децили выше) |

«Модель ничего не предсказывала» — следствие слабого базового классификатора и оценки против прокси-файла,
а не свойство данных. Сильный бустинг + S-learner + честный Qini дают рабочий таргетинг: топ-10% по скору
приносят втрое больший прирост, чем случайная рассылка. Продакшн-скрипт: `backend/training/uplift.py`,
сервинг: `app/uplift.py::SoloModelUplift`.